# N02: VLM Descriptions + Sentence-BERT Retrieval (Experiment 2)

In **Experiment 0**, we proved that searching purely on metadata (*genre*, *style*, *artist*) completely fails on visual concepts (**0.0% Recall**).

In **Experiment 1 & 2**, we test the **Data Bottleneck Hypothesis**:
> By using a compact Vision-Language Model (`Salesforce/blip-image-captioning-base`) to extract visual descriptions and concatenating them with the metadata, we unlock semantic retrieval for visual concepts.

### Strict Protocol (ADR 0003 & 0005)
- **Corpus:** Museum A validation split (480 paintings).
- **Controlled Experiment:** We use the exact same text encoder as E0 (`all-MiniLM-L6-v2`) to isolate the effect of visual text information.
- **Museum B (test):** Locked for final test.

In [ ]:
import pandas as pd
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from deep_learning_project_waerniers_lamy.nlp.utils import load_paintings, evaluate_retrieval_benchmark, save_nlp_result, plot_retrieved_paintings
from deep_learning_project_waerniers_lamy.nlp.benchmark import get_benchmark_queries
from deep_learning_project_waerniers_lamy.nlp.E01_generate_descriptions import load_cached_descriptions

# 1. Load paintings and VLM descriptions
paintings = load_paintings(split="val")
descriptions = load_cached_descriptions()
paintings["description"] = paintings["id"].map(descriptions).fillna("")
print(f"Loaded {len(paintings)} paintings. Captioned: {(paintings['description'] != '').sum()}/{len(paintings)}")
paintings[["id", "genre", "artist", "description"]].head(10)


## 2. Encode VLM Descriptions + Metadata

Format: `"{description}. {genre} by {artist}, {style} style"`.

In [ ]:
def format_searchable_text(row):
    desc = row["description"]
    artist = str(row["artist"]).replace("-", " ")
    style = str(row["style"]).replace("_", " ")
    genre = str(row["genre"]).replace("_", " ")
    meta_part = f"{genre} style {style}" if "unknown" in artist.lower() else f"{genre} by {artist}, {style} style"
    return f"{desc}. {meta_part}" if desc else meta_part

texts = [format_searchable_text(r) for _, r in paintings.iterrows()]
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
print(f"Computed embeddings shape: {tuple(doc_embeddings.shape)}")


## 3. Benchmark evaluation on Museum A (Comparison with E0)

In [ ]:
painting_ids = paintings["id"].tolist()

def retrieve(query: str, top_k: int = 10):
    query_emb = embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    scores = torch.matmul(doc_embeddings, query_emb).cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [painting_ids[i] for i in top_indices]

benchmark = get_benchmark_queries()
eval_benchmark = [q for q in benchmark if q.get("relevant_ids")]
categories = sorted(set(q["category"] for q in eval_benchmark))

print("-" * 65)
print(f"{'Category':<18} | {'Queries':<7} | {'Recall@5':<10} | {'Recall@10':<10} | {'MRR':<8}")
print("-" * 65)
for cat in categories:
    cat_queries = [q for q in eval_benchmark if q["category"] == cat]
    res = evaluate_retrieval_benchmark(retrieve, cat_queries, k_values=(5, 10))
    print(f"{cat:<18} | {res['num_queries']:<7} | {res['recall_at_5']*100:8.1f}% | {res['recall_at_10']*100:8.1f}% | {res['mrr']:8.3f}")
print("-" * 65)


## 4. Visual Inspection of Retrieved Results

Let's inspect what paintings are now retrieved for visual queries.

In [ ]:
query = "a snowy scene with people walking"
top_ids = retrieve(query, top_k=5)
retrieved_df = paintings[paintings["id"].isin(top_ids)].set_index("id").loc[top_ids].reset_index()
plot_retrieved_paintings(query, retrieved_df, top_k=5)
